In [74]:
# 目標設定の進捗をグラフ化し、そのグラフをpng保存するプログラム
# 指定したフォルダのパスの中身のcsvファイルすべてに対して、グラフを作成してpng出力するプログラム（フォルダを指定しないといけない...）
import pandas as pd
import os
import datetime
import matplotlib.pyplot as plt
import japanize_matplotlib
from zoneinfo import ZoneInfo  # Python 3.9以上
from pathlib import Path

import glob
from collections import defaultdict, Counter  # ← Counterを追加
import json

import seaborn as sns
import numpy as np

import re

dates = [
    '20250424', '20250501', '20250508', '20250515',
    '20250522', '20250529', '20250605', '20250612',
    '20250619', '20250626', '20250703', '20250710',
    '20250717', '20250724', '20250731', '20250807'
]

# 定数として定義
# now = '20250424'
# now = '20250501'
# now = '20250508'
# now = '20250515'
# now = '20250522'
# now = '20250529'
# now = '20250605'
# now = '20250612'
# now = '20250619'
# now = '20250626'
# now = '20250703'
# now = '20250710'
# now = '20250717'
# now = '20250724'
# now = '20250731'
# now = '20250807'


for now in dates:
    input_month = str(int(now[4:6]))
    input_day = now[6:8]
    # folder_path = '/home/jovyan/work/shared_downloads/Purpose/20250520(no)'
    folder_path = Path('/home/jovyan/work/shared_downloads/Purpose/' + now)
    OUTPUT_DIR = "/home/jovyan/work/graph_image"
    
    
    
    #残したい行を指定（グループの参加者を絞る）
    #自律性・有能感・関連性を支援したグループ
    rows_to_keep_group1 = [
        # "g2125004@fun.ac.jp",#受講前アンケート未回答
        # "g2125007@fun.ac.jp",#完了科目数が少ない
        "g2125010@fun.ac.jp",
        # "g2125013@fun.ac.jp",#Java移行
        "g2125023@fun.ac.jp",
        "g2125040@fun.ac.jp",
        "g2125044@fun.ac.jp",
        "g2125057@fun.ac.jp",
        # "g2125059@fun.ac.jp",#Java移行
        # "g2125069@fun.ac.jp"#完了科目数が少ない
    ]
    
    
    
    #自律性だけ支援したグループ
    rows_to_keep_group3 = [
        "g2125018@fun.ac.jp",
        "g2125020@fun.ac.jp",
        "g2125021@fun.ac.jp",
        # "g2125025@fun.ac.jp",#Java移行
        "g2125035@fun.ac.jp",
        # "g2125037@fun.ac.jp",#受講後アンケート未回答
        # "g2125038@fun.ac.jp",#Java移行
        # "g2125039@fun.ac.jp",#受講後アンケート未回答
        "g2125041@fun.ac.jp",
        "g2125055@fun.ac.jp"
    ]
    
    
    # (★ 修正)
    # グラフに表示したい学生の「メールアドレス」と「表示名」の対応リストを先に定義します。
    # CSVに登場しない学生（目標未設定の学生）も、ここで定義しておけば名前が表示されます。
    #
    # （！！！重要！！！）
    # （以下の「学生A」などの仮名を、表示したい正しい氏名に書き換えてください）
    # （g2125044@fun.ac.jp も含め、rows_to_keep_group1/3 の全員を定義してください）
    MASTER_EMAIL_TO_NAME_MAP = {
        # Group 1
        # "g2125004@fun.ac.jp": "", 
        # "g2125007@fun.ac.jp": "INUKAI Keitaro", 
        "g2125010@fun.ac.jp": "IWASAKI Seiya",
        # "g2125013@fun.ac.jp": "OTA Takeru",
        "g2125023@fun.ac.jp": "KATO Shunya",
        "g2125040@fun.ac.jp": "SUZUKI Hajime",
        "g2125044@fun.ac.jp": "TAKASUGI Takumi",
        "g2125057@fun.ac.jp": "NAKASHIMA Kazunao",
        # "g2125059@fun.ac.jp": "NAKAMURA Keiryu",
        # "g2125069@fun.ac.jp": "HINO Maasa",
        
        # Group 3
        "g2125018@fun.ac.jp": "OZAKI Kai",
        "g2125020@fun.ac.jp": "OTOWA Ryuichi",
        "g2125021@fun.ac.jp": "OMOTO Etsuki",
        # "g2125025@fun.ac.jp": "KABASHIMA Sion",
        "g2125035@fun.ac.jp": "SATO Ren",
        # "g2125037@fun.ac.jp": "SHIMADA Asahi",
        # "g2125038@fun.ac.jp": "SHIMIZU Kanta",
        # "g2125039@fun.ac.jp": "SUGA Hidehiro",
        "g2125041@fun.ac.jp": "SUDO Kento",
        "g2125055@fun.ac.jp": "TODOME Kenta"
    }
    
    
    # 残したい列だけを指定
    columns_to_keep = [
        "姓",
        "名",
        "メールアドレス",
        "状態",
        "開始日時",
        "受験完了",
        "所要時間"
    ]
    
    number = 0
    i = 0
    
    # 辞書を自動ネストできるようにする
    def nested_dict():
        return defaultdict(nested_dict)
    
    # 最終的な構造をここに格納
    data = nested_dict()
    # 各コースごとにすでに登録されたメールアドレスを記録しておく
    email_set = defaultdict(lambda: defaultdict(set))  # email_set[subject][course_name] に保存
    
    #Testユーザの除去（２回目以降のユーザは古い方を残すようにしたい->あくまでやったかどうかをグラフ化するため、完了済みをできるだけ残したいから）
    for category_dir in folder_path.iterdir():
        if not category_dir.is_dir():
            continue
        for course_dir in category_dir.iterdir():
            # print(course_dir)
            number = 1
            # i = 1            
                
            for csv_file in course_dir.glob("*.csv"):
                # print(f"処理中のファイル: {csv_file.name}")  # ← ここでファイル名を表示
                df = pd.read_csv(csv_file)
                if "keitest4" in str(csv_file):
                    df_filtered = df[df['メールアドレス'].isin(rows_to_keep_group1)]                
                elif "keilabtest7" in str(csv_file):
                    df_filtered = df[df['メールアドレス'].isin(rows_to_keep_group3)]                
                else:
                    print("エラー: CSVファイル名に適切なグループ識別子が含まれていません。")
                    exit()
                # 列を抽出して新しいDataFrameを作成
                df_filtered = df_filtered[columns_to_keep]
                
                # print(df_filtered)
                # print("//////////////////////////////////////////////")
                # #成形済みCSVとして出力
                # df_filtered.to_csv(f"/home/jovyan/work/csv/output_{csv_file.name}.csv", index=False, encoding="utf-8-sig")
                # i = i + 1
                
                #科目ごとにオブジェクト配列に格納する
                subject = category_dir.name
                course_name = course_dir.name
    
                for idx, row in enumerate(df_filtered.to_dict(orient="records")):
                    email = row.get('メールアドレス', '')
                    # 重複チェック
                    if email in email_set[subject][course_name]:
                        print(f"⚠️ 重複検出: {email}（{subject} - {course_name}）はすでに登録済みなのでスキップ")
                        continue  # 重複メールは登録しない（必要ならここを外してもOK）  
                        
                    data[subject][course_name][number] = {
                        '姓': row.get('姓', ''),
                        '名': row.get('名', ''),
                        'メールアドレス': row.get('メールアドレス', ''),
                        '状態': row.get('状態', ''),
                        '開始日時': row.get('開始日時', ''),
                        '受験完了': row.get('受験完了', ''),
                        '所要時間': row.get('所要時間', '')
                    }   
                    email_set[subject][course_name].add(email)  # 登録済みとして記録
                    number = number + 1
    
    #csvが一つもない科目を0としてdataに追加する
    subject_list = [f.name for f in folder_path.iterdir() if f.is_dir()]
    # print(subject_list)
    for subject in subject_list: 
        # print(subject)
        if subject not in data:
            data[subject]
          
    
        
    # print(data) # <-- これじゃ見づらい
    
    # defaultdict → dict に再帰的に変換する関数
    def to_dict(obj):
        if isinstance(obj, defaultdict):
            obj = {k: to_dict(v) for k, v in obj.items()}
        return obj
    
    # JSON風に見やすく出力
    # print(json.dumps(to_dict(data), indent=2, ensure_ascii=False))
    
    
    
    
    
    
    # 各グループごとの進捗をまとめる辞書
    group_progress_complete = {
        "keitest4": defaultdict(int),
        "keilabtest7": defaultdict(int)
    }
    
    group_progress_nocomplete = {
        "keitest4": defaultdict(int),
        "keilabtest7": defaultdict(int)
    }
    
    # グループ別のメールアドレスセット
    group_emails = {
        "keitest4": set(rows_to_keep_group1),
        "keilabtest7": set(rows_to_keep_group3)
    }
    
    # グループ別に進捗を集計
    #初期化処理
    subjects = list(data.keys())
    groups = ['keitest4', 'keilabtest7']
    for group in groups:
        for subject in subjects:
            group_progress_complete[group][subject] = 0
            group_progress_nocomplete[group][subject] = 0
    
    #計算処理
    for subject, courses in data.items():
        # print(subject)
        for course_name, users in courses.items():
            group_key = None
            if "keitest4" in course_name:
                group_key = "keitest4"
            elif "keilabtest7" in course_name:
                group_key = "keilabtest7"
            else:
                continue  # 対象外のコースは無視
    
            complete_count = 0
            nocomplete_count = 0
            for user_info in users.values():
                # print(user_info)
                # 各ユーザーについて：メールアドレスがグループに含まれていて、状態 フラグが "完了" になっていれば、そのユーザーを「完了」とカウントする。
                if user_info['メールアドレス'] in group_emails[group_key] and user_info['状態'] == "終了":
                    complete_count += 1
                if user_info['メールアドレス'] in group_emails[group_key] and user_info['状態'] == "進行中":
                    nocomplete_count += 1
    
            group_progress_complete[group_key][subject] += complete_count  # 科目ごとの完了数を加算
            group_progress_nocomplete[group_key][subject] += nocomplete_count  # 科目ごとの進行中数を加算
    
    print(group_progress_complete)
    print(group_progress_nocomplete)
    
    
    
    
    # (これまでのコードの続き)
    print("\nヒートマップ作成処理を開始します (グループ別・テキスト表示)...")
    
    # 1. 必要な情報の準備
    # (★ 修正)
    # 最初にマスターリストから全員の名前を登録
    email_to_name = MASTER_EMAIL_TO_NAME_MAP.copy()
    
    # (★ 修正)
    # 次に、CSVデータ(data)を走査し、CSVに登場した学生の氏名を「上書き」する
    # (これにより、CSVに記載の氏名を使い、re.subのクレンジングが適用される)
    for subject, courses in data.items():
        for course_name, users in courses.items():
            for user_info in users.values():
                email = user_info.get('メールアドレス')
                
                # (★ 修正) 
                # マスターリストに含まれている場合のみ、名前をCSVの情報で更新
                if email and email in email_to_name: 
                    last_name = user_info.get('姓', '')
                    first_name = user_info.get('名', '')
                    name = re.sub(r'\d+\s*', '', f"{last_name} {first_name}".strip())
                    if name: 
                        email_to_name[email] = name # 既存のキーの値を上書き
    
    print(email_to_name) # マスター名簿とCSVデータで更新された最終的な名簿を出力
    
    
    # 全科目のリスト
    all_subjects = sorted(list(subjects)) # subjectsは計算処理の前に定義済み
    
    # 状態を数値で定義
    STATUS_NOT_STARTED = 0
    STATUS_IN_PROGRESS = 1
    STATUS_COMPLETED = 2
    status_map = {"終了": STATUS_COMPLETED, "進行中": STATUS_IN_PROGRESS}
    
    
    # 出力ディレクトリの確認と作成
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    
    # 2. ヒートマップ作成を関数化 (★ 修正)
    def create_and_save_heatmap(student_email_list, group_name, data, all_subjects, email_to_name):
        """
        指定された生徒リストに基づいてヒートマップを作成し、PNG保存する関数
        (★ ユーザーの追加要望を反映)
        """
        print(f"\n[{group_name}] グループのヒートマップ作成中...")
        
        # (★ 修正)
        # Y軸のラベルを作成 (student_email_list の全員が対象)
        # email_to_name はマスターリストで初期化されているため、
        # .get(email, email) を使っても、全員の氏名（マスター名簿由来）が取得される
        students_display = sorted([
            email_to_name.get(email, email) # .get(key, default) を使用
            for email in student_email_list 
        ])
        
        if not students_display:
            print(f"⚠️ [{group_name}] 対象生徒が0人のため、ヒートマップを作成できません。")
            return
    
        # DataFrameを初期化
        heatmap_data = pd.DataFrame(
            index=students_display, 
            columns=all_subjects, 
            data=STATUS_NOT_STARTED
        )
    
        # dataオブジェクトを走査して、各生徒の各科目の最終状態を入力
        for subject, courses in data.items():
            if subject not in heatmap_data.columns:
                continue 
                
            for course_dir, users in courses.items():
                for user_info in users.values():
                    email = user_info.get('メールアドレス')
                    
                    # (★ 修正) 
                    # この関数に渡されたグループの学生リスト(student_email_list)に含まれない
                    # メールアドレスは、たとえdataに存在しても無視する
                    if email not in student_email_list:
                        continue
                        
                    status_str = user_info.get('状態')
                    
                    # (★ 修正)
                    # email_to_name (マスターリスト由来) から名前を取得
                    display_name = email_to_name.get(email, email)
                    
                    if display_name not in heatmap_data.index:
                        continue 
    
                    current_status_val = heatmap_data.loc[display_name, subject]
                    new_status_val = status_map.get(status_str, STATUS_NOT_STARTED)
                    
                    if new_status_val > current_status_val:
                        heatmap_data.loc[display_name, subject] = new_status_val
    
        # セルに表示するテキストのDataFrameを作成
        text_map = {
            STATUS_NOT_STARTED: "未着手",
            STATUS_IN_PROGRESS: "進行中",
            STATUS_COMPLETED: "完了"
        }
        annot_data = heatmap_data.replace(text_map)
    
        # (★ 修正: 行は昇順、列は降順にソート)
        heatmap_data = heatmap_data.sort_index(axis=0, ascending=True).sort_index(axis=1, ascending=False)
        annot_data = annot_data.sort_index(axis=0, ascending=True).sort_index(axis=1, ascending=False)
    
    
        # 4. ヒートマップの描画と保存
        
        plt.figure(figsize=(10, 6))
        
        N = len(students_display)
    
        heatmap = sns.heatmap(
            heatmap_data,          # 色付け用のデータ (0, 1, 2)
            annot=annot_data,      # 注釈(セル内テキスト)
            fmt='s',               # フォーマット 's' (string)
            cmap='YlGn',           
            linewidths=.5,         
            linecolor='black',     
            cbar=False,            
            annot_kws={"size": 12}  
        )
    
        
        
        heatmap.text(1.02, 1.05, f"N = {N}", 
                     transform=heatmap.transAxes, 
                     fontsize=10, 
                     fontweight='bold',
                     va='top', 
                     ha='right', 
                     color='black')
        
        title_date = f"{now}"
        plt.title(f'目標設定の進捗状況ヒートマップ：Group {group_name} ({title_date})', fontsize=16)
        
        plt.xlabel('科目', fontsize=12)
        plt.ylabel('学生', fontsize=12) 
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        
    
        # 5. PNGファイルとして保存
        output_filepath = OUTPUT_DIR + "/" + now + "/purpose_heatmap_" + group_name + now + ".png"
    
        # (★ 修正: 保存先のサブディレクトリが存在しない可能性があるので作成する)
        os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
    
        try:
            # bbox_inches='tight' で N=XX などの枠外の要素も切らずに保存する
            plt.savefig(output_filepath , bbox_inches='tight') 
            print(f"✅ ヒートマップ(YlGn, N={N})を {output_filepath} に保存しました。")
        except Exception as e:
            print(f"❌ ヒートマップの保存に失敗しました ({group_name}): {e}")
    
        plt.close()
    
    
    # 3. グループごとにヒートマップ作成関数を呼び出す (変更なし)
    
    # グループ1 (keitest4)
    create_and_save_heatmap(
        student_email_list=rows_to_keep_group1,
        group_name="1",
        data=data,
        all_subjects=all_subjects,
        email_to_name=email_to_name
    )
    
    # グループ3 (keilabtest7)
    create_and_save_heatmap(
        student_email_list=rows_to_keep_group3,
        group_name="3",
        data=data,
        all_subjects=all_subjects,
        email_to_name=email_to_name
    )
    
    
    print("\nスクリプトの実行が完了しました。")

{'keitest4': defaultdict(<class 'int'>, {'クラウド・コンピューティング': 1, 'ネットワーク基礎技術': 2, 'プロジェクトマネジメント': 2, 'Linux入門': 3, 'システム開発の基礎': 1, 'アルゴリズム入門': 1, '基礎UML言語': 0, '基礎Javaプログラミング言語': 0}), 'keilabtest7': defaultdict(<class 'int'>, {'クラウド・コンピューティング': 0, 'ネットワーク基礎技術': 1, 'プロジェクトマネジメント': 3, 'Linux入門': 2, 'システム開発の基礎': 0, 'アルゴリズム入門': 1, '基礎UML言語': 0, '基礎Javaプログラミング言語': 0})}
{'keitest4': defaultdict(<class 'int'>, {'クラウド・コンピューティング': 0, 'ネットワーク基礎技術': 0, 'プロジェクトマネジメント': 0, 'Linux入門': 0, 'システム開発の基礎': 0, 'アルゴリズム入門': 0, '基礎UML言語': 0, '基礎Javaプログラミング言語': 0}), 'keilabtest7': defaultdict(<class 'int'>, {'クラウド・コンピューティング': 0, 'ネットワーク基礎技術': 0, 'プロジェクトマネジメント': 0, 'Linux入門': 0, 'システム開発の基礎': 0, 'アルゴリズム入門': 0, '基礎UML言語': 0, '基礎Javaプログラミング言語': 0})}

ヒートマップ作成処理を開始します (グループ別・テキスト表示)...
{'g2125010@fun.ac.jp': 'IWASAKI Seiya', 'g2125023@fun.ac.jp': 'KATO Shunya', 'g2125040@fun.ac.jp': 'SUZUKI Hajime', 'g2125044@fun.ac.jp': 'TAKASUGI Takumi', 'g2125057@fun.ac.jp': 'NAKASHIMA Kazunao', 'g2125018@fun.ac.jp': 'OZAKI Kai', '